> ## SOLUTION / ANSWER KEY &mdash; Lab 9.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-cite-every-claim.ipynb`](../lab-02-cite-every-claim.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.2 &mdash; Cite Every Claim

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Build a claim that carries its statement, value AND the exact source string
- Assemble a multi-claim summary and detect which claims are uncited
- See why one uncited claim in a mix breaks auditability

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Auditability: structure & the trail](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-02"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Lab 1 checked a *single* figure was grounded. A real summary makes **many** claims at once, and the
danger is the **mix**: five cited numbers and one silently uncited one. Auditability means every conclusion
is **traceable** (deck slide 15), so each **claim** carries the exact **source string** it came from
&mdash; the citation a later validation step (Lab 7) checks for *correctness*. Here you build the claim
record and a detector that names **which** claims in a summary are uncited.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("a claim carries the SOURCE STRING through, e.g.:")
print({"statement": "revenue", "value": 120.0, "source": "p4, income stmt"})
print("a summary is a LIST of these -- and we must find any that are uncited.")

## Build it
Complete `make_claim` (carry the source string through) and `uncited_claims` (name every claim in a
summary that is missing a citation &mdash; the mix detector), then run the cell.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

def make_claim(statement, fig):
    # a claim ties a statement to its grounded value AND carries its exact source string
    return {"statement": statement, "value": fig["value"], "source": fig["source"]}

def uncited_claims(claims):
    # return the STATEMENT of each claim in the summary that is missing a source citation
    return [c["statement"] for c in claims if not c.get("source")]

try:
    c = make_claim("revenue", REPORT["revenue"])
    print("claim:", c)
    summary = [make_claim("revenue", REPORT["revenue"]),
               make_claim("net_income", REPORT["net_income"]),
               {"statement": "guess", "value": 5.0, "source": ""}]   # a slipped-in uncited claim
    print("uncited in the mix:", uncited_claims(summary))
    print("fully-cited pair  :", uncited_claims(summary[:2]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A claim carries its **exact source string** &mdash; not "cited: yes", but the page the number came from.
- The mix detector names the **one** uncited claim among several cited ones &mdash; the failure that's easy to miss.
- Lab 7 will check these citations for *correctness* (right page); here we just guarantee each claim *has* one.

## Your turn (open task &mdash; no grader)
Build a three-claim summary where the middle claim has `source=None`, and confirm `uncited_claims`
returns just that one. **What good looks like:** the detector pinpoints exactly the uncited claim(s), so a
single slipped-in figure can never ride along uncited into an analyst's summary.

---
### Key takeaway
A summary is a mix of claims, and one silently uncited number breaks the chain. Carrying the exact source string through -- and naming which claims lack it -- is what a validator (Lab 7) checks and what makes the agent auditable.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>